# Natural Language Goals — End-to-End Tutorial

This tutorial combines the NL → GoalSpec interpreter with the full GOAP execution
loop. Instead of manually constructing `GoalSpec` objects, users describe their
goals in plain English and LangGOAP handles the rest.

## What This Covers

| Step | Description |
|------|-------------|
| Define actions | Data pipeline with `fetch_data`, `clean_data`, `generate_report` |
| Manual GoalSpec | The "old way" — explicit conditions, constraints, objectives |
| NL GoalSpec | Same result via `interpreter.interpret("Generate a report")` |
| Full execution | `GoapGraph.invoke(goal=interpreted_goal)` |
| One-liner | `GoapGraph.invoke_nl("Generate a report", llm=llm)` |
| CSP constraints | NL budget limits flow through to the CSP optimizer |

## Define Actions

A three-step data pipeline with real execute callables and resource estimates.

In [ ]:
from typing import Any

from langgoap import ActionSpec


def fetch_data(ws: dict[str, Any]) -> dict[str, Any]:
    """Simulate fetching raw data."""
    return {"data_fetched": True, "raw_data": ["row1", "row2", "row3"]}


def clean_data(ws: dict[str, Any]) -> dict[str, Any]:
    """Simulate cleaning data."""
    raw = ws.get("raw_data", [])
    return {"data_clean": True, "clean_data": [r.upper() for r in raw]}


def generate_report(ws: dict[str, Any]) -> dict[str, Any]:
    """Simulate report generation."""
    data = ws.get("clean_data", [])
    return {"report_complete": True, "report": f"Report: {len(data)} records processed"}


actions = [
    ActionSpec(
        name="fetch_data",
        effects={"data_fetched": True},
        execute=fetch_data,
        resources={"cost_usd": 0.5, "api_calls": 1},
        metadata={"description": "Fetch raw data from the source"},
    ),
    ActionSpec(
        name="clean_data",
        preconditions={"data_fetched": True},
        effects={"data_clean": True},
        execute=clean_data,
        resources={"cost_usd": 0.1},
        metadata={"description": "Validate and clean the raw data"},
    ),
    ActionSpec(
        name="generate_report",
        preconditions={"data_clean": True},
        effects={"report_complete": True},
        execute=generate_report,
        resources={"cost_usd": 1.0, "tokens": 500},
        metadata={"description": "Generate a formatted report from clean data"},
    ),
]

print(f"Defined {len(actions)} actions: {[a.name for a in actions]}")

### GOAP Execution Graph

The planner discovers a plan, the executor runs each action, and the
observer checks progress — replanning automatically if something fails.

In [ ]:
from IPython.display import Image, display

graph = GoapGraph(actions=actions)
display(Image(graph.compile().get_graph().draw_mermaid_png()))

## Manual GoalSpec (The Old Way)

Without the interpreter, you construct `GoalSpec` objects explicitly.

In [ ]:
from langgoap import GoalSpec, GoapGraph

manual_goal = GoalSpec(conditions={"report_complete": True})
result = GoapGraph(actions=actions).invoke(goal=manual_goal, world_state={})

print(f"Status: {result['status']}")
print(f"Report: {result['world_state']['report']}")

In [ ]:
result["plan"].visualize()

## NL GoalSpec (The New Way)

The interpreter produces the same `GoalSpec` from plain English.

In [ ]:
from langchain_openai import ChatOpenAI

from langgoap import GoalInterpreter

llm = ChatOpenAI(model="gpt-4o-mini")
interpreter = GoalInterpreter(llm=llm, actions=actions)

nl_goal = interpreter.interpret("Generate a report from the data")
print(f"Interpreted GoalSpec: {nl_goal}")
print(f"Conditions: {dict(nl_goal.conditions)}")

## Full Execution with NL Goal

Pass the interpreted goal to the GOAP graph.

In [ ]:
result = GoapGraph(actions=actions).invoke(goal=nl_goal, world_state={})

print(f"Status: {result['status']}")
print(f"Report: {result['world_state']['report']}")
print(f"Actions: {[h.action_name for h in result['execution_history'] if h.success]}")

## One-Liner Convenience

`invoke_nl()` combines interpretation and execution in a single call.

In [ ]:
result = GoapGraph(actions=actions).invoke_nl(
    "Generate a report from the raw data",
    llm=llm,
)

print(f"Status: {result['status']}")
print(f"Report: {result['world_state']['report']}")

## With CSP Constraints

Budget language in the NL request flows through to the CSP optimizer.
The interpreter extracts constraints, and the planner validates resource
usage against them.

In [ ]:
from langgoap import CSPStatus

# Interpret with budget constraint
goal_with_budget = interpreter.interpret("Generate a report, keeping total cost under $10")
print(f"Constraints: {goal_with_budget.constraints}")

result = GoapGraph(actions=actions).invoke(goal=goal_with_budget, world_state={})

print(f"\nStatus: {result['status']}")
plan = result.get("plan")
if plan and plan.metadata.csp:
    csp = plan.metadata.csp
    print(f"CSP status: {csp.status}")
    for ru in csp.resource_usage:
        print(f"  {ru.key}: used={ru.used}, limit={ru.limit}, satisfied={ru.satisfied}")

## Inspecting LLM Reasoning

Use `interpret_raw()` to see the LLM's explanation of how it
derived the goal — useful for debugging and building trust.

In [ ]:
raw = interpreter.interpret_raw(
    "Generate the cheapest report possible, with total cost under $5"
)

print(f"Reasoning: {raw.reasoning}")
print(f"Conditions: {raw.conditions}")
print(f"Constraints: {raw.constraints}")
print(f"Objectives: {raw.objectives}")